In [1]:
import pandas as pd
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras import layers, Model

In [2]:
# my notes

# follow up on climatology precipitation & temperature
# challenges
# entity embeddings
# decrease selection model
# no self-attention mechanism
# full data

# Data Loading & Preprocessing

## loading climatology precipitation

In [3]:
climo_df = pd.read_csv('precip_climo.csv')

In [4]:
# Ensure time is datetime
climo_df["time"] = pd.to_datetime(climo_df["time"])

# Filter for years 1982–1984
climo_df = climo_df[
    (climo_df["time"].dt.year >= 1982) &
    (climo_df["time"].dt.year <= 1984)
]

climo_df

,latitude,longitude,time,precip_climo
87412,-10.0,-70.00,1982-01-03,17.455254
87413,-10.0,-69.75,1982-01-03,20.454031
87414,-10.0,-69.50,1982-01-03,21.781550
87415,-10.0,-69.25,1982-01-03,23.039633
87416,-10.0,-69.00,1982-01-03,24.171701
...,...,...,...,...
351324,0.0,-61.00,1984-12-30,18.504717
351325,0.0,-60.75,1984-12-30,19.413860
351326,0.0,-60.50,1984-12-30,18.435364
351327,0.0,-60.25,1984-12-30,17.345550


## loading amazon

In [5]:
df = pd.read_csv('amazon_1982_1984.csv')

In [6]:
df = df[(df['latitude'] <= 0) & (df['latitude'] >= -10) & (df['longitude'] <= -60) & (df['longitude'] >= -70)].reset_index(drop=True)

In [7]:
df.columns

Index(['week', 'latitude', 'longitude', 'precip_max', 'z_mean', 'z_min',
       'z_max', 'z_std', 'z_p25', 'z_p75', 't2m_mean', 't2m_min', 't2m_max',
       't2m_std', 'swvl1_mean', 'swvl1_min', 'swvl1_max', 'swvl1_std'],
      dtype='object')

In [8]:
df[(df['latitude']==-10.0) &(df['longitude']==-70.00)]

,week,latitude,longitude,precip_max,z_mean,z_min,z_max,z_std,z_p25,z_p75,t2m_mean,t2m_min,t2m_max,t2m_std,swvl1_mean,swvl1_min,swvl1_max,swvl1_std
0,1982-01-03,-10.0,-70.0,53.920998,57299.777,57094.934,57484.700,97.39884,57224.979492,57381.106445,297.094976,294.97217,303.80005,1.553519,0.477107,0.402939,0.520004,0.024311
1681,1982-01-10,-10.0,-70.0,19.614357,57295.977,57036.260,57584.523,130.51627,57198.779297,57380.193359,297.110008,294.50415,302.78564,2.047954,0.474737,0.397247,0.520004,0.025554
3362,1982-01-17,-10.0,-70.0,21.611408,57420.887,57214.117,57616.200,102.87246,57331.707031,57504.681641,298.137471,294.72876,303.98170,2.356913,0.471536,0.420258,0.520004,0.014697
5043,1982-01-24,-10.0,-70.0,59.675793,57445.840,57174.863,57700.945,122.44274,57354.431641,57535.673828,297.612650,294.83690,303.36328,2.063976,0.478310,0.413406,0.520004,0.018787
6724,1982-01-31,-10.0,-70.0,27.553007,57421.730,57216.188,57656.480,97.11432,57352.368164,57492.026367,298.166442,294.92432,304.40137,2.475648,0.472525,0.440613,0.520004,0.013322
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
305943,1984-12-02,-10.0,-70.0,13.217761,57297.000,57136.140,57478.266,104.86479,57194.148438,57387.768555,297.970376,294.59375,303.56860,2.412716,0.473545,0.401917,0.520004,0.013066
309304,1984-12-09,-10.0,-70.0,52.271850,57329.810,57116.586,57523.902,102.65150,57237.964844,57415.329102,297.340830,293.99707,303.19214,1.920816,0.479858,0.405304,0.520004,0.017691
310985,1984-12-16,-10.0,-70.0,16.090420,57263.920,57026.870,57518.098,118.33471,57162.831055,57341.011719,298.072969,294.54395,304.60498,2.656609,0.472368,0.409424,0.520004,0.016673
312666,1984-12-23,-10.0,-70.0,20.289717,57224.547,56979.590,57474.074,109.87511,57138.765625,57306.655273,298.016097,294.31860,304.24536,2.530852,0.453458,0.399277,0.520004,0.024784


In [9]:
climo_df = climo_df.rename(columns={"time": "week"})

In [10]:
climo_df["week"] = pd.to_datetime(climo_df["week"])
df["week"] = pd.to_datetime(df["week"])

In [11]:
df = pd.merge(
    df,
    climo_df,
    on=["latitude", "longitude", "week"],
    how="inner"
)

In [12]:
print(df.shape)
print(df.head())

(316028, 19)
        week  latitude  longitude  precip_max     z_mean      z_min     z_max  \
0 1982-01-03     -10.0     -70.00   53.920998  57299.777  57094.934  57484.70   
1 1982-01-03     -10.0     -69.75   52.655270  57299.234  57092.934  57481.20   
2 1982-01-03     -10.0     -69.50   54.742490  57303.430  57097.434  57482.45   
3 1982-01-03     -10.0     -69.25   53.261463  57303.168  57097.684  57481.95   
4 1982-01-03     -10.0     -69.00   55.323917  57306.160  57101.684  57486.70   

      z_std         z_p25         z_p75    t2m_mean    t2m_min    t2m_max  \
0  97.39884  57224.979492  57381.106445  297.094976  294.97217  303.80005   
1  96.57474  57225.404297  57380.173828  297.150437  294.80664  301.67114   
2  95.29932  57231.651367  57386.486328  297.289216  295.03320  302.14966   
3  94.47691  57229.719727  57389.490234  297.482480  295.50195  304.60670   
4  94.23981  57233.131836  57391.543945  297.389318  295.46484  303.31372   

    t2m_std  swvl1_mean  swvl1_min  s

In [13]:
df.columns

Index(['week', 'latitude', 'longitude', 'precip_max', 'z_mean', 'z_min',
       'z_max', 'z_std', 'z_p25', 'z_p75', 't2m_mean', 't2m_min', 't2m_max',
       't2m_std', 'swvl1_mean', 'swvl1_min', 'swvl1_max', 'swvl1_std',
       'precip_climo'],
      dtype='object')

In [14]:
# ============================
# 0. Hyperparameters
# ============================

HIST_WINDOW = 12      # number of past weeks (encoder length)
FUTURE_HORIZON = 12   # number of future weeks (decoder length)

# ============================
# 1. Define column groups
# ============================

print("✅ [Step 1] Defining column groups...")

# Static features (do NOT change). These will be label-encoded later.
static_cols = ['latitude', 'longitude']

# Historical inputs (past 12 weeks) - includes month as a feature
hist_cols = [
    'month',                # categorical but treated as numeric feature
    'precip_max',
    'swvl1_mean','swvl1_min','swvl1_max','swvl1_std',
    'z_mean','z_min','z_max','z_std','z_p25','z_p75',
    't2m_mean','t2m_min','t2m_max','t2m_std',
    'precip_climo'
]

# Future known reals (next 12 weeks)
future_real_cols = [
    't2m_mean','t2m_min','t2m_max','t2m_std', 'precip_climo'
]

# Future known categoricals (next 12 weeks)
future_cat_cols = ['month']

# Combined future inputs in time dimension
future_cols = future_cat_cols + future_real_cols

target_col = 'precip_max'

# Categorical columns that will be label-encoded
cat_cols = ['month', 'latitude', 'longitude']

print("   static_cols:", static_cols)
print("   hist_cols  :", hist_cols)
print("   future_cols:", future_cols)
print("   target_col :", target_col)


# ============================
# 2. Sort & time handling
# ============================

print("\n✅ [Step 2] Sorting df and creating 'week' + 'month'...")

# Assumes you already have a DataFrame named df
# df columns include: week, latitude, longitude, precip_max, t2m_*, swvl1_*, z_*, etc.

df = df.sort_values(['latitude', 'longitude', 'week']).reset_index(drop=True)

df['week'] = pd.to_datetime(df['week'])
df['month'] = df['week'].dt.month

print("   df shape after sort:", df.shape)
print("   week range:", df['week'].min(), "→", df['week'].max())


# ============================
# 3. Label-encode categoricals
# ============================

print("\n✅ [Step 3] Label-encoding categorical columns (month, latitude, longitude)...")

label_encoders = {}
for c in cat_cols:
    le = LabelEncoder()
    df[c] = le.fit_transform(df[c])
    label_encoders[c] = le
    print(f"   Encoded {c}: {len(le.classes_)} unique values")

# NOTE:
# - 'month' is now an integer ID (0..11)
# - 'latitude' and 'longitude' are integer IDs (for static embeddings / static inputs)


# ============================
# 4. Train / test split in time
# ============================

print("\n✅ [Step 4] Holding out last 12 weeks globally...")

max_week = df['week'].max()
cutoff = max_week - pd.Timedelta(weeks=FUTURE_HORIZON)

train_df = df[df['week'] <= cutoff].copy()
test_df  = df[df['week'] > cutoff].copy()

print("   Cutoff date :", cutoff)
print("   train_df shape:", train_df.shape)
print("   test_df shape :", test_df.shape)


# ============================
# 5. Sliding window builder
# ============================

print("\n✅ [Step 5] Building sliding window function...")

def build_windows(group, k=HIST_WINDOW, tau=FUTURE_HORIZON):
    """
    group: DataFrame for a single (lat, lon) series, sorted by week.

    Returns:
        X_hist:   (n_samples, k,   len(hist_cols))
        X_future:(n_samples, tau,  len(future_cols))
        X_static:(n_samples,       len(static_cols))
        y:       (n_samples,)
    """
    group = group.reset_index(drop=True)

    X_hist, X_future, X_static, y = [], [], [], []

    # we need k past points and tau future points, so t runs:
    # t = k, ..., len(group) - tau - 1
    for t in range(k, len(group) - tau):
        # Historical window: [t-k, ..., t-1]
        hist_block = group.loc[t-k:t-1, hist_cols].values

        # Future known window: [t, ..., t+tau-1]
        fut_block  = group.loc[t:t+tau-1, future_cols].values

        # Static inputs taken at time t (or any index, since static)
        static_vec = group.loc[t, static_cols].values  # (len(static_cols),)

        # Target at the horizon end: time t+tau-1
        target_val = group.loc[t+tau-1, target_col]

        X_hist.append(hist_block)
        X_future.append(fut_block)
        X_static.append(static_vec)
        y.append(target_val)

    return (
        np.array(X_hist),
        np.array(X_future),
        np.array(X_static),
        np.array(y)
    )

print("   Window builder ready.")


# ============================
# 6. Build windows per grid point
# ============================

print("\n✅ [Step 6] Building windows for each (latitude, longitude) group...")

all_hist, all_fut, all_stat, all_y = [], [], [], []

group_count = 0
total_samples = 0

for (lat_id, lon_id), g in train_df.groupby(['latitude', 'longitude']):
    group_count += 1
    print(f"   ▶ Group {group_count}: (lat_id={lat_id}, lon_id={lon_id}), rows={len(g)}")

    Xh, Xf, Xs, yy = build_windows(g)

    if len(yy) == 0:
        print("     ⚠ Skipped (not enough history + future)")
        continue

    print(f"     Created {len(yy)} samples for this group.")

    total_samples += len(yy)
    all_hist.append(Xh)
    all_fut.append(Xf)
    all_stat.append(Xs)
    all_y.append(yy)

print("\n   Total groups processed :", group_count)
print("   Total samples created  :", total_samples)


# ============================
# 7. Stack into final tensors
# ============================

print("\n✅ [Step 7] Stacking all group windows into final numpy arrays...")

X_hist   = np.vstack(all_hist)    # (N, HIST_WINDOW, len(hist_cols))
X_future = np.vstack(all_fut)     # (N, FUTURE_HORIZON, len(future_cols))
X_static = np.vstack(all_stat)    # (N, len(static_cols))
y        = np.concatenate(all_y)  # (N,)

print("✅ [FINAL SHAPES]")
print("   X_hist  :", X_hist.shape)    # (N, 12, 17)
print("   X_future:", X_future.shape)  # (N, 12, 6)
print("   X_static:", X_static.shape)  # (N, 2)
print("   y       :", y.shape)         # (N,)


# ============================
# 8. Quick sanity check
# ============================

print("\n✅ [Step 8] Sanity check on first sample:")

print("   Past months (first sample):")
print("   ", X_hist[0, :, 0])  # month over past 12 weeks

print("   Future months (first sample):")
print("   ", X_future[0, :, 0])  # month over next 12 weeks

print("   Static (lat_id, lon_id):")
print("   ", X_static[0])

print("   Target y (precip_max at t+12):")
print("   ", y[0])

print("\n🎉 Preprocessing pipeline finished successfully.")

✅ [Step 1] Defining column groups...
   static_cols: ['latitude', 'longitude']
   hist_cols  : ['month', 'precip_max', 'swvl1_mean', 'swvl1_min', 'swvl1_max', 'swvl1_std', 'z_mean', 'z_min', 'z_max', 'z_std', 'z_p25', 'z_p75', 't2m_mean', 't2m_min', 't2m_max', 't2m_std', 'precip_climo']
   future_cols: ['month', 't2m_mean', 't2m_min', 't2m_max', 't2m_std', 'precip_climo']
   target_col : precip_max

✅ [Step 2] Sorting df and creating 'week' + 'month'...
   df shape after sort: (316028, 20)
   week range: 1982-01-03 00:00:00 → 1984-12-30 00:00:00

✅ [Step 3] Label-encoding categorical columns (month, latitude, longitude)...
   Encoded month: 12 unique values
   Encoded latitude: 41 unique values
   Encoded longitude: 41 unique values

✅ [Step 4] Holding out last 12 weeks globally...
   Cutoff date : 1984-10-07 00:00:00
   train_df shape: (292494, 20)
   test_df shape : (23534, 20)

✅ [Step 5] Building sliding window function...
   Window builder ready.

✅ [Step 6] Building windows for e

# GRN

In [15]:
@keras.utils.register_keras_serializable(package="tft")
class GRN(layers.Layer):
    """
    Simple Gated Residual Network (GRN) with optional context.

    - input a: (..., d_out)
    - if use_context=True, call as grn(a, c)
    - if use_context=False, call as grn(a)

    Output has same last dimension d_out so that residual a + x is valid.
    """

    def __init__(self, hidden_units, output_dim, dropout=0.0, use_context=False, **kwargs):
        super().__init__(**kwargs)
        self.hidden_units = hidden_units
        self.output_dim = output_dim
        self.dropout_rate = dropout
        self.use_context = use_context

        # define sub-layers directly; no custom build needed
        self.dense1 = layers.Dense(hidden_units, activation="elu")
        self.dense2 = layers.Dense(output_dim)

        self.gate_dense = layers.Dense(2 * output_dim)
        self.dropout = layers.Dropout(dropout)
        self.layer_norm = layers.LayerNormalization(epsilon=1e-6)

    def call(self, a, c=None, training=None):
        """
        a: (..., output_dim)
        c: (..., d_c) or None
        """
        if self.use_context:
            if c is None:
                raise ValueError("GRN created with use_context=True: call as grn(a, c)")
            x = tf.concat([a, c], axis=-1)
        else:
            if c is not None:
                raise ValueError("GRN created with use_context=False: call as grn(a)")
            x = a

        # main MLP
        x = self.dense1(x)
        x = self.dense2(x)
        x = self.dropout(x, training=training)

        # gating (GLU)
        gate_in = self.gate_dense(x)           # (..., 2 * d_out)
        linear, gate = tf.split(gate_in, 2, axis=-1)
        x = linear * tf.sigmoid(gate)          # (..., d_out)

        # residual + norm
        y = self.layer_norm(a + x)             # (..., d_out)
        return y

# Variable selection

In [16]:
@keras.utils.register_keras_serializable(package="tft")
class VariableSelectionNetwork(layers.Layer):
    """
    Variable Selection Network (VSN).

    Input:
        x:        (B, F)
        context:  (B, d_ctx) or None

    Output:
        selected: (B, d_model)
        weights:  (B, F)
    """

    def __init__(self,
                 num_features,
                 d_model,
                 hidden_units,
                 use_context=False,
                 dropout=0.0,
                 name=None):
        super().__init__(name=name)
        self.num_features = num_features
        self.d_model = d_model
        self.hidden_units = hidden_units
        self.use_context = use_context
        self.dropout = dropout

        # one embedding per feature: scalar -> d_model
        self.feature_embedders = [
            layers.Dense(d_model, name=f"embed_feat_{i}")
            for i in range(num_features)
        ]

        # one GRN per feature (no context), output dim = d_model
        self.feature_grns = [
            GRN(hidden_units=hidden_units,
                output_dim=d_model,
                dropout=dropout,
                use_context=False,
                name=f"grn_feat_{i}")
            for i in range(num_features)
        ]

        # GRN for computing selection logits from flattened embeddings:
        # input dim = F * d_model, output dim = F * d_model (for residual)
        self.flat_dim = num_features * d_model
        self.flatten_grn = GRN(
            hidden_units=hidden_units,
            output_dim=self.flat_dim,
            dropout=dropout,
            use_context=use_context,
            name="flatten_grn"
        )

        # final linear layer -> F logits
        self.weight_dense = layers.Dense(num_features, name="weight_dense")

    def call(self, x, context=None, training=None):
        """
        x: (B, F)
        context: (B, d_ctx) or None
        """
        B = tf.shape(x)[0]

        # 1. Embed each feature: (B, 1) -> (B, d_model)
        embedded_feats = []
        for i, emb in enumerate(self.feature_embedders):
            xi = x[:, i:i+1]     # (B, 1)
            ei = emb(xi)        # (B, d_model)
            embedded_feats.append(ei)

        embedded = tf.stack(embedded_feats, axis=1)  # (B, F, d_model)

        # 2. Per-feature GRN transform (no context)
        transformed_feats = []
        for i, grn in enumerate(self.feature_grns):
            ti = grn(embedded_feats[i], training=training)   # (B, d_model)
            transformed_feats.append(ti)

        transformed = tf.stack(transformed_feats, axis=1)   # (B, F, d_model)

        # 3. Compute feature weights
        flat_emb = tf.reshape(embedded, [B, self.flat_dim])  # (B, F*d_model)

        if self.use_context:
            if context is None:
                raise ValueError("VSN created with use_context=True but context=None.")
            flat_out = self.flatten_grn(flat_emb, context, training=training)
        else:
            if context is not None:
                raise ValueError("VSN created with use_context=False but context was provided.")
            flat_out = self.flatten_grn(flat_emb, training=training)

        logits = self.weight_dense(flat_out)          # (B, F)
        weights = tf.nn.softmax(logits, axis=-1)      # (B, F)

        # 4. Weighted sum
        w_expanded = tf.expand_dims(weights, axis=-1) # (B, F, 1)
        weighted = transformed * w_expanded           # (B, F, d_model)
        selected = tf.reduce_sum(weighted, axis=1)    # (B, d_model)

        return selected, weights

# TFT

In [17]:
@keras.utils.register_keras_serializable(package="tft")
class TFT(Model):
    """
    Temporal Fusion Transformer (simplified) with:
      - static variable selection (1 VSN)
      - shared historical VSN (applied across all lookback steps)
      - shared future VSN (applied across all lookforward steps)
      - LSTM encoder (historical) and decoder (future)
      - Dense head → 3 quantile predictions

    Inputs:
        x_hist:   (B, T_enc, F_hist)   e.g. (B, 12, 17)
        x_future: (B, T_dec, F_future) e.g. (B, 12,  6)
        x_static: (B, F_static)        e.g. (B, 2)

    Outputs dict:
        preds:            (B, num_quantiles)

        static_selected:  (B, d_model)
        static_weights:   (B, F_static)

        hist_selected:    (B, T_enc, d_model)
        hist_weights:     (B, T_enc, F_hist)

        future_selected:  (B, T_dec, d_model)
        future_weights:   (B, T_dec, F_future)

        encoder_output:   (B, T_enc, lstm_units)
        decoder_output:   (B, T_dec, lstm_units)
    """

    def __init__(
        self,
        lookback_steps,        # T_enc (e.g. 12)
        lookforward_steps,     # T_dec (e.g. 12)
        num_hist_features,     # 17
        num_future_features,   # 6
        num_static_features,   # 2
        d_model=64,
        vsn_hidden_units=64,
        lstm_units=64,
        dropout=0.1,
        num_quantiles=3,
        **kwargs,
    ):
        super().__init__(**kwargs)

        self.lookback_steps = lookback_steps
        self.lookforward_steps = lookforward_steps
        self.num_hist_features = num_hist_features
        self.num_future_features = num_future_features
        self.num_static_features = num_static_features
        self.d_model = d_model
        self.num_quantiles = num_quantiles

        # fixed quantiles for quantile loss
        self.quantiles = tf.constant([0.1, 0.5, 0.9], dtype=tf.float32)

        # ---------- Static variable selection ----------
        self.vsn_static = VariableSelectionNetwork(
            num_features=num_static_features,
            d_model=d_model,
            hidden_units=vsn_hidden_units,
            use_context=False,
            dropout=dropout,
            name="vsn_static",
        )

        # ---------- Shared historical VSN (with static context) ----------
        self.vsn_hist = VariableSelectionNetwork(
            num_features=num_hist_features,
            d_model=d_model,
            hidden_units=vsn_hidden_units,
            use_context=True,   # receives static_selected as context
            dropout=dropout,
            name="vsn_hist_shared",
        )

        # ---------- Shared future VSN (no context) ----------
        self.vsn_future = VariableSelectionNetwork(
            num_features=num_future_features,
            d_model=d_model,
            hidden_units=vsn_hidden_units,
            use_context=False,
            dropout=dropout,
            name="vsn_future_shared",
        )

        # ---------- LSTMs ----------
        self.encoder_lstm = layers.LSTM(
            lstm_units,
            return_sequences=True,
            return_state=True,
            name="encoder_lstm",
        )

        self.decoder_lstm = layers.LSTM(
            lstm_units,
            return_sequences=True,
            return_state=True,
            name="decoder_lstm",
        )

        # ---------- Output head (decoder → 3 values) ----------
        self.output_layer = layers.Dense(
            num_quantiles, name="quantile_head"
        )

    # --------------------------------------------------
    # Quantile loss (same as before)
    # --------------------------------------------------
    def quantile_loss(self, y_true, preds):
        """
        y_true: (B,) or (B, 1)
        preds:  (B, Q) where Q = num_quantiles (3)

        QL(y, y_hat, q) = q (y - y_hat)_+ + (1-q) (y_hat - y)_+
        """
        y_true = tf.reshape(y_true, (-1, 1))      # (B, 1)
        errors = y_true - preds                   # (B, Q)

        relu_pos = tf.nn.relu(errors)            # (B, Q)
        relu_neg = tf.nn.relu(-errors)           # (B, Q)

        q = tf.reshape(self.quantiles, (1, -1))  # (1, Q)

        loss_per_q = q * relu_pos + (1.0 - q) * relu_neg  # (B, Q)
        loss = tf.reduce_mean(loss_per_q)                 # scalar
        return loss

    # --------------------------------------------------
    # Forward pass
    # --------------------------------------------------
    def call(self, inputs, training=None):
        x_hist, x_future, x_static = inputs

        B = tf.shape(x_hist)[0]
        T_enc = self.lookback_steps
        T_dec = self.lookforward_steps
        F_hist = self.num_hist_features
        F_future = self.num_future_features

        # ==========================================================
        # 1. Static variable selection (no time dimension)
        # ==========================================================
        static_selected, static_weights = self.vsn_static(
            x_static, training=training
        )  # (B, d_model), (B, F_static)

        # ==========================================================
        # 2. Historical VSN (shared), applied across all time steps
        # ==========================================================
        # x_hist: (B, T_enc, F_hist)
        # repeat static context for each encoder time step
        static_ctx_time = tf.tile(
            tf.expand_dims(static_selected, axis=1),  # (B, 1, d_model)
            [1, T_enc, 1],
        )  # (B, T_enc, d_model)

        # flatten time → (B*T_enc, F_hist) and context → (B*T_enc, d_model)
        x_hist_flat = tf.reshape(x_hist, [B * T_enc, F_hist])
        ctx_flat = tf.reshape(static_ctx_time, [B * T_enc, self.d_model])

        hist_sel_flat, hist_w_flat = self.vsn_hist(
            x_hist_flat, context=ctx_flat, training=training
        )  # (B*T_enc, d_model), (B*T_enc, F_hist)

        # reshape back to sequences
        hist_selected = tf.reshape(hist_sel_flat, [B, T_enc, self.d_model])
        hist_weights  = tf.reshape(hist_w_flat,   [B, T_enc, F_hist])

        # ==========================================================
        # 3. Future VSN (shared), applied across all future steps
        # ==========================================================
        # x_future: (B, T_dec, F_future)
        x_future_flat = tf.reshape(x_future, [B * T_dec, F_future])

        future_sel_flat, future_w_flat = self.vsn_future(
            x_future_flat, training=training
        )  # (B*T_dec, d_model), (B*T_dec, F_future)

        future_selected = tf.reshape(future_sel_flat, [B, T_dec, self.d_model])
        future_weights  = tf.reshape(future_w_flat,   [B, T_dec, F_future])

        # ==========================================================
        # 4. LSTM encoder/decoder
        # ==========================================================
        enc_output, state_h, state_c = self.encoder_lstm(
            hist_selected, training=training
        )  # (B, T_enc, lstm_units), (B, lstm_units)*2

        dec_output, _, _ = self.decoder_lstm(
            future_selected,
            initial_state=[state_h, state_c],
            training=training,
        )  # (B, T_dec, lstm_units)

        # ==========================================================
        # 5. Output head: last decoder step → 3 predictions
        # ==========================================================
        last_step = dec_output[:, -1, :]      # (B, lstm_units)
        preds = self.output_layer(last_step)  # (B, num_quantiles)

        return {
            "preds": preds,
            "static_selected": static_selected,
            "static_weights": static_weights,
            "hist_selected": hist_selected,
            "hist_weights": hist_weights,
            "future_selected": future_selected,
            "future_weights": future_weights,
            "encoder_output": enc_output,
            "decoder_output": dec_output,
        }

In [18]:
lambda_smooth = 0.05  # tune this (e.g. try 0.01, 0.05, 0.1)

def spatial_smoothness_loss(preds, x_static, eps=1e-6):
    """
    preds   : (B, Q) -> TFT quantile predictions per sample
    x_static: (B, 2) -> [latitude_id, longitude_id] (label-encoded ints)

    We use the median quantile (index 1) to enforce spatial smoothness:
        L_smooth = mean_{(i,j) neighbors} (y_hat_i - y_hat_j)^2
    where neighbors differ by ±1 in encoded latitude OR ±1 in encoded longitude.
    """
    # pick median quantile
    y_hat = preds[:, 1]  # (B,)

    lat = tf.cast(x_static[:, 0:1], tf.float32)  # (B,1)
    lon = tf.cast(x_static[:, 1:2], tf.float32)  # (B,1)

    # Pairwise differences in encoded ID space
    dlat = lat - tf.transpose(lat)  # (B,B)
    dlon = lon - tf.transpose(lon)  # (B,B)

    # Adjacent in latitude: difference = ±1
    up_down = (
        tf.abs(dlat - 1.0) < eps
    ) | (
        tf.abs(dlat + 1.0) < eps
    )

    # Adjacent in longitude: difference = ±1
    left_right = (
        tf.abs(dlon - 1.0) < eps
    ) | (
        tf.abs(dlon + 1.0) < eps
    )

    neighbor_mask = tf.logical_or(up_down, left_right)

    # Remove self-pairs (i == j)
    B = tf.shape(lat)[0]
    eye = tf.eye(B, dtype=tf.bool)
    neighbor_mask = tf.logical_and(neighbor_mask, tf.logical_not(eye))

    # Pairwise prediction differences
    diff  = tf.expand_dims(y_hat, 1) - tf.expand_dims(y_hat, 0)  # (B,B)
    diff2 = tf.square(diff)

    # Keep only neighbor pairs
    diff2_neighbors = tf.boolean_mask(diff2, neighbor_mask)

    smooth_loss = tf.cond(
        tf.size(diff2_neighbors) > 0,
        lambda: tf.reduce_mean(diff2_neighbors),
        lambda: tf.constant(0.0, dtype=tf.float32),
    )
    return smooth_loss

# Training

In [19]:
print(X_hist.shape)
print(X_future.shape)
print(X_static.shape)
print(y.shape)

(252150, 12, 17)
(252150, 12, 6)
(252150, 2)
(252150,)


In [25]:
tft = TFT(
    lookback_steps=HIST_WINDOW,
    lookforward_steps=FUTURE_HORIZON,
    num_hist_features=len(hist_cols),      # 17
    num_future_features=len(future_cols),  # 6
    num_static_features=len(static_cols),  # 2 (latitude_id, longitude_id)
    d_model=64,
    vsn_hidden_units=64,
    lstm_units=64,
    dropout=0.1,
    num_quantiles=3,
)

optimizer = tf.keras.optimizers.Adam(learning_rate=1e-3)

In [26]:
X_hist_tf   = tf.convert_to_tensor(X_hist,   dtype=tf.float32)
X_future_tf = tf.convert_to_tensor(X_future, dtype=tf.float32)
X_static_tf = tf.convert_to_tensor(X_static, dtype=tf.float32)

In [27]:
test_out = tft(
    (X_hist_tf[:4], X_future_tf[:4], X_static_tf[:4]),
    training=False
)
print(test_out["preds"].shape)  # should be (4, 3)

(4, 3)


In [28]:
# ensure float32
X_hist_tf   = tf.convert_to_tensor(X_hist,   dtype=tf.float32)
X_future_tf = tf.convert_to_tensor(X_future, dtype=tf.float32)
X_static_tf = tf.convert_to_tensor(X_static, dtype=tf.float32)
y_tf        = tf.convert_to_tensor(y,        dtype=tf.float32)

batch_size = 256
epochs = 10

dataset = tf.data.Dataset.from_tensor_slices(
    ((X_hist_tf, X_future_tf, X_static_tf), y_tf)
)
dataset = dataset.shuffle(buffer_size=10000).batch(batch_size).prefetch(tf.data.AUTOTUNE)

steps_per_epoch = int(np.ceil(X_hist.shape[0] / batch_size))
print("Steps per epoch:", steps_per_epoch)

Steps per epoch: 985


In [29]:
@tf.function
def train_step(x_hist_b, x_future_b, x_static_b, y_b):
    with tf.GradientTape() as tape:
        outputs = tft((x_hist_b, x_future_b, x_static_b), training=True)
        preds = outputs["preds"]  # (B, 3) for 3 quantiles

        base_loss   = tft.quantile_loss(y_b, preds)
        smooth_loss = spatial_smoothness_loss(preds, x_static_b)

        loss = base_loss + lambda_smooth * smooth_loss

    grads = tape.gradient(loss, tft.trainable_variables)
    optimizer.apply_gradients(zip(grads, tft.trainable_variables))
    return base_loss, smooth_loss, loss


# ============================
# Training loop
# ============================

with open("training_smooth_ver.log", "w", encoding="utf-8") as log_file:

    for epoch in range(epochs):
        msg = f"\nEpoch {epoch+1}/{epochs}"
        print(msg)
        log_file.write(msg + "\n")
        log_file.flush()

        epoch_loss = 0.0

        for step, ((x_hist_b, x_future_b, x_static_b), y_b) in enumerate(dataset):
            base_l, smooth_l, loss = train_step(
                x_hist_b, x_future_b, x_static_b, y_b
            )

            loss_val   = float(loss.numpy())
            base_val   = float(base_l.numpy())
            smooth_val = float(smooth_l.numpy())

            epoch_loss += loss_val

            if (step + 1) % 50 == 0 or (step + 1) == steps_per_epoch:
                avg_loss = epoch_loss / (step + 1)

                msg = (
                    f"Step {step+1:4d}/{steps_per_epoch} | "
                    f"batch_loss={loss_val:.4f} | "
                    f"base={base_val:.4f} | "
                    f"smooth={smooth_val:.4f} | "
                    f"avg_loss={avg_loss:.4f}"
                )

                print(" ", msg)
                log_file.write(msg + "\n")
                log_file.flush()


Epoch 1/10
  Step   50/985 | batch_loss=4.4888 | base=4.4888 | smooth=0.0005 | avg_loss=5.6394
  Step  100/985 | batch_loss=4.5508 | base=4.5508 | smooth=0.0001 | avg_loss=5.0042
  Step  150/985 | batch_loss=4.0610 | base=4.0610 | smooth=0.0002 | avg_loss=4.7040
  Step  200/985 | batch_loss=3.7871 | base=3.7740 | smooth=0.2621 | avg_loss=4.5157
  Step  250/985 | batch_loss=3.7044 | base=3.6842 | smooth=0.4033 | avg_loss=4.3789
  Step  300/985 | batch_loss=3.6748 | base=3.6503 | smooth=0.4909 | avg_loss=4.2484
  Step  350/985 | batch_loss=2.9671 | base=2.9458 | smooth=0.4263 | avg_loss=4.1165
  Step  400/985 | batch_loss=2.9099 | base=2.8912 | smooth=0.3732 | avg_loss=3.9979
  Step  450/985 | batch_loss=2.7737 | base=2.7594 | smooth=0.2846 | avg_loss=3.8956
  Step  500/985 | batch_loss=3.1649 | base=3.1520 | smooth=0.2579 | avg_loss=3.8077
  Step  550/985 | batch_loss=2.9167 | base=2.9020 | smooth=0.2934 | avg_loss=3.7210
  Step  600/985 | batch_loss=2.9075 | base=2.8958 | smooth=0.233

In [30]:
# tft.save("tft_model.keras")              # FULL safe save (REQUIRED)
tft.save_weights("smooth_tft_weights.weights.h5")

In [31]:
def build_test_windows(group, cutoff_week, k=HIST_WINDOW, tau=FUTURE_HORIZON):
    """
    Build windows for test, using the FULL group (not pre-cut to train_df),
    but only keep samples whose TARGET horizon lies AFTER cutoff_week
    (i.e., in the last 12 weeks).

    group: DataFrame for a single (lat, lon) series, sorted by week.

    Returns:
        X_hist_test:   (n_test, k,   len(hist_cols))
        X_future_test: (n_test, tau, len(future_cols))
        X_static_test: (n_test,      len(static_cols))
        y_test:        (n_test,)
        target_weeks:  (n_test,)  # for debugging / inspection
    """
    group = group.sort_values("week").reset_index(drop=True)

    X_hist, X_future, X_static, y, target_weeks = [], [], [], [], []

    # same range pattern as training:
    # t = k, ..., len(group) - tau - 1
    for t in range(k, len(group) - tau):
        horizon_idx = t + tau - 1
        horizon_week = group.loc[horizon_idx, "week"]

        # only keep windows whose TARGET is in the hold-out period
        if horizon_week <= cutoff_week:
            continue

        # Historical window: [t-k, ..., t-1]
        hist_block = group.loc[t-k:t-1, hist_cols].values

        # Future known window: [t, ..., t+tau-1]
        fut_block  = group.loc[t:t+tau-1, future_cols].values

        # Static inputs at time t
        static_vec = group.loc[t, static_cols].values  # (len(static_cols),)

        # Target at horizon: precip_max at t+tau-1
        target_val = group.loc[horizon_idx, target_col]

        X_hist.append(hist_block)
        X_future.append(fut_block)
        X_static.append(static_vec)
        y.append(target_val)
        target_weeks.append(horizon_week)

    return (
        np.array(X_hist),
        np.array(X_future),
        np.array(X_static),
        np.array(y),
        np.array(target_weeks),
    )


In [32]:
print("\n✅ [TEST] Building windows whose horizon is in last 12 weeks...")

X_hist_test_list   = []
X_future_test_list = []
X_static_test_list = []
y_test_list        = []
week_test_list     = []

test_group_count = 0
test_total_samples = 0

for (lat_id, lon_id), g in df.groupby(['latitude', 'longitude']):
    test_group_count += 1
    print(f"   ▶ Test group {test_group_count}: (lat_id={lat_id}, lon_id={lon_id}), rows={len(g)}")

    Xh_t, Xf_t, Xs_t, yy_t, weeks_t = build_test_windows(
        g, cutoff_week=cutoff, k=HIST_WINDOW, tau=FUTURE_HORIZON
    )

    if len(yy_t) == 0:
        print("     ⚠ Skipped (no horizons in last 12 weeks)")
        continue

    print(f"     Created {len(yy_t)} TEST samples for this group.")
    print(f"     Horizon weeks from {weeks_t.min()} to {weeks_t.max()}")

    test_total_samples += len(yy_t)
    X_hist_test_list.append(Xh_t)
    X_future_test_list.append(Xf_t)
    X_static_test_list.append(Xs_t)
    y_test_list.append(yy_t)
    week_test_list.append(weeks_t)

print("\n   Total test groups used   :", test_group_count)
print("   Total TEST samples       :", test_total_samples)

# Stack
X_hist_test   = np.vstack(X_hist_test_list)    # (N_test, 12, len(hist_cols))
X_future_test = np.vstack(X_future_test_list)  # (N_test, 12, len(future_cols))
X_static_test = np.vstack(X_static_test_list)  # (N_test, len(static_cols))
y_test        = np.concatenate(y_test_list)    # (N_test,)
weeks_test    = np.concatenate(week_test_list) # (N_test,)

print("\n✅ [TEST FINAL SHAPES]")
print("   X_hist_test  :", X_hist_test.shape)
print("   X_future_test:", X_future_test.shape)
print("   X_static_test:", X_static_test.shape)
print("   y_test       :", y_test.shape)
print("   weeks_test   :", weeks_test.shape)
print("   Horizon week range:", weeks_test.min(), "→", weeks_test.max())


✅ [TEST] Building windows whose horizon is in last 12 weeks...
   ▶ Test group 1: (lat_id=0, lon_id=0), rows=188
     Created 13 TEST samples for this group.
     Horizon weeks from 1984-10-14 00:00:00 to 1984-12-23 00:00:00
   ▶ Test group 2: (lat_id=0, lon_id=1), rows=188
     Created 13 TEST samples for this group.
     Horizon weeks from 1984-10-14 00:00:00 to 1984-12-23 00:00:00
   ▶ Test group 3: (lat_id=0, lon_id=2), rows=188
     Created 13 TEST samples for this group.
     Horizon weeks from 1984-10-14 00:00:00 to 1984-12-23 00:00:00
   ▶ Test group 4: (lat_id=0, lon_id=3), rows=188
     Created 13 TEST samples for this group.
     Horizon weeks from 1984-10-14 00:00:00 to 1984-12-23 00:00:00
   ▶ Test group 5: (lat_id=0, lon_id=4), rows=188
     Created 13 TEST samples for this group.
     Horizon weeks from 1984-10-14 00:00:00 to 1984-12-23 00:00:00
   ▶ Test group 6: (lat_id=0, lon_id=5), rows=188
     Created 13 TEST samples for this group.
     Horizon weeks from 1984-10

In [33]:
X_hist_test_tf   = tf.convert_to_tensor(X_hist_test,   dtype=tf.float32)
X_future_test_tf = tf.convert_to_tensor(X_future_test, dtype=tf.float32)
X_static_test_tf = tf.convert_to_tensor(X_static_test, dtype=tf.float32)
y_test_tf        = tf.convert_to_tensor(y_test,        dtype=tf.float32)

In [34]:
test_outputs = tft(
    (X_hist_test_tf, X_future_test_tf, X_static_test_tf),
    training=False
)
y_test_pred = test_outputs["preds"].numpy()   # (N_test, 3)

In [35]:
pred_df = pd.DataFrame(
    y_test_pred,
    columns=["q10", "q50", "q90"]
)

# optional: add horizon week for reference
pred_df["horizon_week"] = weeks_test

pred_df.to_csv("y_test_pred_smooth.csv", index=False)

print("Saved to y_test_pred_smooth.csv")

Saved to y_test_pred_smooth.csv


In [36]:
tft_new = TFT(
    lookback_steps=12,
    lookforward_steps=12,
    num_hist_features=17,
    num_future_features=6,
    num_static_features=2,
    d_model=64,
    vsn_hidden_units=64,
    lstm_units=64,
    dropout=0.1,
    num_quantiles=3,
)

# Shape signatures: (batch, T_enc, F_hist), (batch, T_dec, F_future), (batch, F_static)
tft_new.build([
    (None, 12, 17),
    (None, 12, 6),
    (None, 2),
])

tft_new.load_weights("tft_weights.weights.h5")
print("✅ Weights loaded.")

C:\Users\Windows 11\anaconda3\Lib\site-packages\keras\src\layers\layer.py:421: UserWarning: `build()` was called on layer 'tft_2', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(


✅ Weights loaded.
